In [ ]:
# ============================================================
# 10_Feature_Engineering.ipynb
# Section 1 — Setup + Build Historical Features (no leakage)
# ============================================================
# Ziel / الهدف:
#   Give the tabular models (LR, XGBoost) HISTORICAL features
#   that summarize each engine's past, and compare 3 levels of
#   information: last value / last-50 / full-history.
#   This answers the professor's fairness concern with an EXPERIMENT.
# ============================================================

import os, sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")
sys.path.insert(0, SRC_DIR)
from preprocessing import FINAL_FEATURES_FD001
N_SENSORS = len(FINAL_FEATURES_FD001)
print(f"Sensors used: {N_SENSORS}  ->  {FINAL_FEATURES_FD001}")

# --- Metric (identical to all notebooks) ---
def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0) - 1.0,
                                         np.exp(d/10.0) - 1.0)))

# --- Load the FROZEN processed data (same as all models) ---
train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
y_true = test_last["RUL_true"].values
units_test = test_last["unit"].values
print(f"\nTrain rows: {len(train)} | Test engines: {test_full['unit'].nunique()}")

# ============================================================
# FEATURE BUILDER — summarizes a window of cycles into features
# ============================================================
# For each sensor we compute 5 summary features:
#   mean, std, slope (linear trend), last value, delta (last-first)
# لكل سينسور: المتوسط، التشتت، الميل، آخر قيمة، الفرق الكلي
def summarize_window(window_values):
    """window_values: array (n_cycles, n_sensors) -> feature vector."""
    n = window_values.shape[0]
    feats = []
    x = np.arange(n)
    for j in range(window_values.shape[1]):
        col = window_values[:, j]
        mean = col.mean()
        std  = col.std()
        last = col[-1]
        delta = col[-1] - col[0]
        # slope = linear trend (least-squares) over the window
        if n >= 2 and np.std(x) > 0:
            slope = np.polyfit(x, col, 1)[0]
        else:
            slope = 0.0
        feats.extend([mean, std, slope, last, delta])
    return np.array(feats, dtype=np.float32)

FEATURE_NAMES = []
for s in FINAL_FEATURES_FD001:
    FEATURE_NAMES += [f"{s}_mean", f"{s}_std", f"{s}_slope", f"{s}_last", f"{s}_delta"]

# ============================================================
# BUILD THE THREE INPUT VERSIONS
# ============================================================
# A) last value only  (current baseline: 13 features)
# B) summary of LAST 50 cycles   (13 x 5 = 65 features)
# C) summary of FULL history     (13 x 5 = 65 features)
#
# NO LEAKAGE: for TEST engines, the window is only the cycles that
# were actually observed (up to the last available cycle) — never
# future cycles. Same evaluation point as all other models.
# ============================================================
def build_version(df_train, df_test_full, units_order, mode):
    """mode: 'A_last', 'B_win50', 'C_full'"""
    # ---- TRAIN: one sample per row would be huge; to match the
    #      other models (last-cycle evaluation), we build ONE sample
    #      per (engine, cycle) using history UP TO that cycle.
    #      For fairness with test, we train on the SAME summary form.
    Xtr, ytr = [], []
    for unit, g in df_train.groupby("unit"):
        g = g.sort_values("cycle")
        feats_all = g[FINAL_FEATURES_FD001].values
        labels = g["RUL_piecewise125"].values
        L = len(g)
        for end in range(L):
            if mode == "A_last":
                w = feats_all[end:end+1]
            elif mode == "B_win50":
                start = max(0, end - 49)
                w = feats_all[start:end+1]
            else:  # C_full
                w = feats_all[0:end+1]
            Xtr.append(summarize_window(w) if mode != "A_last"
                       else feats_all[end])
            ytr.append(labels[end])
    # ---- TEST: one sample per engine at its LAST observed cycle
    Xte = []
    for unit in units_order:
        g = df_test_full[df_test_full["unit"] == unit].sort_values("cycle")
        feats_all = g[FINAL_FEATURES_FD001].values
        if mode == "A_last":
            Xte.append(feats_all[-1])
        elif mode == "B_win50":
            Xte.append(summarize_window(feats_all[-50:]))
        else:  # C_full
            Xte.append(summarize_window(feats_all))
    return (np.array(Xtr, dtype=np.float32), np.array(ytr, dtype=np.float32),
            np.array(Xte, dtype=np.float32))

print("\nBuilding the three input versions (A/B/C)...")
data = {}
for mode in ["A_last", "B_win50", "C_full"]:
    Xtr, ytr, Xte = build_version(train, test_full, units_test, mode)
    data[mode] = (Xtr, ytr, Xte)
    print(f"  {mode:8s}: train {Xtr.shape}, test {Xte.shape}")

print("\n✓ Section 1 complete: features built with no leakage.")
print("  A = last value only (baseline)")
print("  B = summary of last 50 cycles")
print("  C = summary of full history")

In [ ]:
# ============================================================
# 10_Feature_Engineering.ipynb. # SUPERSEDED — test-based;
# Section 2 — Train LR + XGBoost on A/B/C, compare fairly
# ============================================================
# Same fair protocol: same 13 sensors, piecewise-125 label,
# scaling fit on TRAIN only (no leakage). We compare each model
# across the 3 input versions, and against the recorded LSTM.
# ============================================================
from sklearn.preprocessing import MinMaxScaler

# XGBoost config = IDENTICAL to Notebook 04 (the selected config)
XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=42, n_jobs=-1, objective="reg:squarederror")

VERSION_LABELS = {
    "A_last":  "A: last value only (baseline)",
    "B_win50": "B: summary of last 50 cycles",
    "C_full":  "C: summary of full history",
}

def evaluate(model_name, Xtr, ytr, Xte, y_true):
    # Scale features: fit on TRAIN only (no leakage) — same rule as always
    scaler = MinMaxScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)

    if model_name == "LR":
        model = LinearRegression().fit(Xtr_s, ytr)
        y_pred = model.predict(Xte_s)
    else:  # XGBoost (trees don't need scaling, but we keep it uniform/harmless)
        model = xgb.XGBRegressor(**XGB_PARAMS).fit(Xtr, ytr)
        y_pred = model.predict(Xte)

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ns   = nasa_score(y_true, y_pred)
    return mae, rmse, ns, y_pred

# --- Run all combinations ---
rows = []
preds_store = {}
print("=" * 70)
print("RESULTS — LR & XGBoost across the three input versions (piecewise)")
print("=" * 70)
print(f"{'Model':<9} {'Version':<32} {'MAE':>7} {'RMSE':>7} {'NASA':>9}")
print("-" * 70)
for model_name in ["LR", "XGBoost"]:
    for mode in ["A_last", "B_win50", "C_full"]:
        Xtr, ytr, Xte = data[mode]
        mae, rmse, ns, y_pred = evaluate(model_name, Xtr, ytr, Xte, y_true)
        rows.append({"model": model_name, "version": mode,
                     "mae": round(mae, 2), "rmse": round(rmse, 2),
                     "nasa": round(ns, 2)})
        preds_store[(model_name, mode)] = y_pred
        print(f"{model_name:<9} {VERSION_LABELS[mode]:<32} "
              f"{mae:>7.2f} {rmse:>7.2f} {ns:>9.2f}")
    print("-" * 70)

results_fe = pd.DataFrame(rows)

# --- Reference values (recorded, piecewise) for context ---
print("\nReference (recorded results, piecewise):")
print(f"  LR   (A, from Notebook 03):   MAE 17.79")
print(f"  XGB  (A, from Notebook 04):   MAE 13.21")
print(f"  LSTM (sequence, Notebook 05): MAE 10.69   <- the target to approach")
print(f"  Chronos (Notebook 06):        MAE 12.15")

# --- The key question answered ---
print("\n" + "=" * 70)
print("KEY QUESTION — does historical information close the gap?")
print("=" * 70)
lr_A  = results_fe[(results_fe.model=="LR") & (results_fe.version=="A_last")]["mae"].values[0]
lr_C  = results_fe[(results_fe.model=="LR") & (results_fe.version=="C_full")]["mae"].values[0]
xgb_A = results_fe[(results_fe.model=="XGBoost") & (results_fe.version=="A_last")]["mae"].values[0]
xgb_C = results_fe[(results_fe.model=="XGBoost") & (results_fe.version=="C_full")]["mae"].values[0]
print(f"  LR:      {lr_A:.2f} (last only)  ->  {lr_C:.2f} (full history)   "
      f"= {100*(lr_A-lr_C)/lr_A:+.1f}% change")
print(f"  XGBoost: {xgb_A:.2f} (last only)  ->  {xgb_C:.2f} (full history)   "
      f"= {100*(xgb_A-xgb_C)/xgb_A:+.1f}% change")
print(f"\n  LSTM (sequence model): 10.69")
print(f"  Gap XGBoost(full) vs LSTM: {xgb_C - 10.69:+.2f} cycles")

# --- Save ---
fe_path = os.path.join(RESULTS_DIR, "fd001_feature_engineering.csv")
results_fe.to_csv(fe_path, index=False)
print(f"\nSaved: {fe_path}")

In [ ]:
# ============================================================
# 10_Feature_Engineering.ipynb
# Section 3 — Visualize the fairness result + save
# ============================================================
import matplotlib.pyplot as plt

FIGURES_DIR = os.path.join(PROJECT_ROOT, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

# --- Figure: MAE across information levels, LR vs XGBoost, vs LSTM ---
versions = ["A_last", "B_win50", "C_full"]
x_labels = ["A\nlast value", "B\nlast 50 cycles", "C\nfull history"]
x = np.arange(len(versions))

lr_mae  = [results_fe[(results_fe.model=="LR") & (results_fe.version==v)]["mae"].values[0] for v in versions]
xgb_mae = [results_fe[(results_fe.model=="XGBoost") & (results_fe.version==v)]["mae"].values[0] for v in versions]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x, lr_mae,  "o-", color="#999999", linewidth=2, markersize=9, label="Linear Regression")
ax.plot(x, xgb_mae, "s-", color="#2E75B6", linewidth=2, markersize=9, label="XGBoost")
ax.axhline(10.69, color="#C00000", linestyle="--", linewidth=2,
           label="LSTM (sequence model) = 10.69")

# annotate each point
for xi, v in zip(x, lr_mae):
    ax.annotate(f"{v:.2f}", (xi, v), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=9, color="#666666")
for xi, v in zip(x, xgb_mae):
    ax.annotate(f"{v:.2f}", (xi, v), textcoords="offset points",
                xytext=(0, -16), ha="center", fontsize=9, color="#2E75B6", fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(x_labels)
ax.set_xlabel("Information given to the tabular model")
ax.set_ylabel("MAE (cycles) — lower is better")
ax.set_title("Fair comparison: giving tabular models historical features\n"
             "XGBoost with full history (8.57) outperforms the LSTM (10.69)")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
ax.invert_yaxis()  # lower MAE = higher on chart (better = up)
plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "fd001_feature_engineering.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

# --- Closing summary ---
print("\n" + "=" * 60)
print("NOTEBOOK 10 COMPLETE — FAIR COMPARISON")
print("=" * 60)
print("""
Key finding: when tabular models receive historical features
summarizing each engine's full past (mean, std, slope, last,
delta per sensor), XGBoost reaches MAE 8.57 — BETTER than the
LSTM (10.69). The earlier advantage of sequence models is
largely explained by INFORMATION ACCESS, not architecture.

Artifacts:
  results/fd001_feature_engineering.csv
  figures/fd001_feature_engineering.png
""")

In [ ]:
# ============================================================
# NB10 — Section 4: Save per-engine predictions LR(C) & XGB(C)
# (needed for the fair Wilcoxon test in NB08)
# ============================================================
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
import xgboost as xgb

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")

# needs data["C_full"], units_test, y_true from Section 1 of this notebook
Xtr_C, ytr_C, Xte_C = data["C_full"]

XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=42, n_jobs=-1, objective="reg:squarederror")

# LR(C)
scaler = MinMaxScaler().fit(Xtr_C)
lr = LinearRegression().fit(scaler.transform(Xtr_C), ytr_C)
pred_lr_C = np.clip(lr.predict(scaler.transform(Xte_C)), 0, 125)

# XGBoost(C)
xgbm = xgb.XGBRegressor(**XGB_PARAMS).fit(Xtr_C, ytr_C)
pred_xgb_C = np.clip(xgbm.predict(Xte_C), 0, 125)

# Save per-engine predictions (fair, full-history version C)
pd.DataFrame({"unit": units_test, "RUL_true": y_true,
              "pred_LR_full": pred_lr_C}).to_csv(
    os.path.join(RESULTS_DIR, "fd001_lr_full_history.csv"), index=False)
pd.DataFrame({"unit": units_test, "RUL_true": y_true,
              "pred_XGB_full": pred_xgb_C}).to_csv(
    os.path.join(RESULTS_DIR, "fd001_xgb_full_history.csv"), index=False)

print("Saved fair per-engine predictions:")
print(f"  LR(C):  MAE {np.mean(np.abs(pred_lr_C - y_true)):.2f}  (expect 14.99)")
print(f"  XGB(C): MAE {np.mean(np.abs(pred_xgb_C - y_true)):.2f}  (expect 8.57)")

In [ ]:
# ============================================================
# NB10 — Section 5: Permutation Feature Importance (XGBoost, C)
# بنثبت: أنهي features بتفرق أكتر (slope? mean? std?)
# العدالة: permutation — كل feature تتبعثر بنفس الطريقة، مفيش تحيّز
# ============================================================
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")
FIGURES_DIR = os.path.join(DATA_DIR, "..", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

# Rebuild XGBoost(C) — same config as everywhere
Xtr_C, ytr_C, Xte_C = data["C_full"]
XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=42, n_jobs=-1, objective="reg:squarederror")
xgbm = xgb.XGBRegressor(**XGB_PARAMS).fit(Xtr_C, ytr_C)

# Baseline error on test
base_pred = np.clip(xgbm.predict(Xte_C), 0, 125)
base_mae = mean_absolute_error(y_true, base_pred)
print(f"Baseline MAE (no permutation): {base_mae:.3f}\n")

# --- Permutation importance: shuffle each feature, measure MAE rise ---
rng = np.random.default_rng(42)
N_REPEATS = 10
importances = np.zeros(Xte_C.shape[1])
for j in range(Xte_C.shape[1]):
    rises = []
    for _ in range(N_REPEATS):
        Xp = Xte_C.copy()
        rng.shuffle(Xp[:, j])                      # break feature j only
        mae_p = mean_absolute_error(y_true, np.clip(xgbm.predict(Xp), 0, 125))
        rises.append(mae_p - base_mae)
    importances[j] = np.mean(rises)

imp_df = pd.DataFrame({"feature": FEATURE_NAMES,
                       "importance": importances}).sort_values(
                       "importance", ascending=False).reset_index(drop=True)

# --- Show top 15 ---
print("Top 15 most important features (MAE rise when shuffled):")
print(imp_df.head(15).to_string(index=False))

# --- Aggregate by statistic type (mean/std/slope/last/delta) ---
imp_df["stat"] = imp_df["feature"].str.split("_").str[-1]
by_stat = imp_df.groupby("stat")["importance"].sum().sort_values(ascending=False)
print("\nTotal importance by statistic type:")
for stat, val in by_stat.items():
    print(f"  {stat:>6}: {val:.3f}")

# --- Plot top 15 ---
fig, ax = plt.subplots(figsize=(9, 7))
top = imp_df.head(15).iloc[::-1]
ax.barh(top["feature"], top["importance"], color="#2E75B6", edgecolor="black")
ax.set_xlabel("Importance (MAE increase when feature is shuffled)")
ax.set_title("XGBoost — Top 15 feature importances (permutation)\n"
             "Fair full-history model")
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "fd001_xgb_feature_importance.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

imp_df.to_csv(os.path.join(RESULTS_DIR, "fd001_xgb_feature_importance.csv"), index=False)
print(f"\nSaved: fd001_xgb_feature_importance.csv + figure")

In [ ]:
# ============================================================
# NB10 — Section 6: Smoothing test (fair)
# Goal: does smoothing the signal before feature extraction
#       improve XGBoost, or do std/slope features already
#       capture the information?
# Fairness: identical XGBoost(C); only difference = smoothing.
# No leakage: causal rolling mean (past values only).
# ============================================================
import os
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")
SMOOTH_WINDOW = 5   # causal moving-average window
XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=42, n_jobs=-1, objective="reg:squarederror")

# Causal smoothing: rolling mean over PAST values only (no future leakage)
def smooth_engine(feats, window):
    df = pd.DataFrame(feats)
    return df.rolling(window=window, min_periods=1).mean().values.astype(np.float32)

# Build version C (full-history summary) on SMOOTHED signals
def build_smoothed_C(df_train, df_test_full, units_order, window):
    Xtr, ytr = [], []
    for unit, g in df_train.groupby("unit"):
        g = g.sort_values("cycle")
        feats = g[FINAL_FEATURES_FD001].values.astype(np.float32)
        feats_sm = smooth_engine(feats, window)
        labels = g["RUL_piecewise125"].values
        for end in range(len(g)):
            Xtr.append(summarize_window(feats_sm[0:end+1]))
            ytr.append(labels[end])
    Xte = []
    for unit in units_order:
        g = df_test_full[df_test_full["unit"] == unit].sort_values("cycle")
        feats = g[FINAL_FEATURES_FD001].values.astype(np.float32)
        feats_sm = smooth_engine(feats, window)
        Xte.append(summarize_window(feats_sm))
    return (np.array(Xtr, np.float32), np.array(ytr, np.float32),
            np.array(Xte, np.float32))

print(f"Building smoothed version C (causal MA, window={SMOOTH_WINDOW})...")
Xtr_sm, ytr_sm, Xte_sm = build_smoothed_C(train, test_full, units_test, SMOOTH_WINDOW)

# Train XGBoost on smoothed features
xgb_sm = xgb.XGBRegressor(**XGB_PARAMS).fit(Xtr_sm, ytr_sm)
pred_sm = np.clip(xgb_sm.predict(Xte_sm), 0, 125)

mae_sm  = mean_absolute_error(y_true, pred_sm)
rmse_sm = np.sqrt(mean_squared_error(y_true, pred_sm))
ns_sm   = nasa_score(y_true, pred_sm)

# Compare: raw (recorded) vs smoothed
print("\n" + "=" * 60)
print("SMOOTHING TEST — XGBoost(full history), raw vs smoothed")
print("=" * 60)
print(f"{'Version':<20}{'MAE':>8}{'RMSE':>8}{'NASA':>10}")
print("-" * 60)
print(f"{'Raw (recorded)':<20}{8.69:>8.2f}{12.13:>8.2f}{241.29:>10.2f}")
print(f"{'Smoothed (MA=5)':<20}{mae_sm:>8.2f}{rmse_sm:>8.2f}{ns_sm:>10.2f}")
print("-" * 60)

diff = mae_sm - 8.69
if diff < -0.1:
    print(f"\n  Smoothing HELPED: MAE {8.69:.2f} -> {mae_sm:.2f} ({diff:+.2f})")
    print("  -> smoothing reveals the degradation trend more clearly.")
elif diff > 0.1:
    print(f"\n  Smoothing HURT: MAE {8.69:.2f} -> {mae_sm:.2f} ({diff:+.2f})")
    print("  -> std/slope features already carry the signal;")
    print("     smoothing removes useful variance information.")
else:
    print(f"\n  Smoothing had little effect: MAE {8.69:.2f} -> {mae_sm:.2f} ({diff:+.2f})")
    print("  -> the features already handle noise; smoothing not needed.")

pd.DataFrame([
    {"version": "raw",      "mae": 8.69, "rmse": 12.13, "nasa": 241.29},
    {"version": "smoothed", "mae": round(mae_sm, 2), "rmse": round(rmse_sm, 2),
     "nasa": round(ns_sm, 2)},
]).to_csv(os.path.join(RESULTS_DIR, "fd001_smoothing_test.csv"), index=False)
print(f"\nSaved: fd001_smoothing_test.csv")

In [ ]:
# ============================================================
# NB10 — Section 7: Non-linear shape features (supervisor's idea)
# She suggested: a parameter describing the CURVE SHAPE
# (x² vs 2x²) beyond the linear slope.
# Added per sensor: quad  = quadratic coefficient (curvature)
#                   ratio = slope(last third)/slope(first third)
# Test: identical XGBoost, 65 vs 91 features. Only change = features.
# ============================================================
import numpy as np, pandas as pd, xgboost as xgb, os, time
from sklearn.metrics import mean_absolute_error, mean_squared_error

def summarize_ext(w):
    """(n_cycles, 13) -> 91 features: original 5 + quad + slope_ratio."""
    n = w.shape[0]; x = np.arange(n); feats = []
    third = max(2, n // 3)
    for j in range(w.shape[1]):
        c = w[:, j]
        mean, std, last, delta = c.mean(), c.std(), c[-1], c[-1] - c[0]
        slope = np.polyfit(x, c, 1)[0] if n >= 2 else 0.0
        quad  = np.polyfit(x, c, 2)[0] if n >= 3 else 0.0   # curvature
        if n >= 2 * third:                                   # acceleration
            s1 = np.polyfit(np.arange(third), c[:third],  1)[0]
            s2 = np.polyfit(np.arange(third), c[-third:], 1)[0]
            ratio = s2 / s1 if abs(s1) > 1e-6 else 0.0
            ratio = float(np.clip(ratio, -50, 50))
        else:
            ratio = 0.0
        feats.extend([mean, std, slope, last, delta, quad, ratio])
    return np.array(feats, dtype=np.float32)

# --- Build train / test with the extended summary ---
t0 = time.time()
Xtr_ext, ytr_ext = [], []
for u, g in train.groupby("unit"):
    g = g.sort_values("cycle")
    f = g[FINAL_FEATURES_FD001].values
    lab = g["RUL_piecewise125"].values
    for end in range(len(g)):
        Xtr_ext.append(summarize_ext(f[:end + 1]))
        ytr_ext.append(lab[end])
Xtr_ext = np.array(Xtr_ext); ytr_ext = np.array(ytr_ext)

Xte_ext = []
for u in units_test:
    g = test_full[test_full["unit"] == u].sort_values("cycle")
    Xte_ext.append(summarize_ext(g[FINAL_FEATURES_FD001].values))
Xte_ext = np.array(Xte_ext)
print(f"Built {Xtr_ext.shape} in {time.time() - t0:.0f}s")

# --- Same XGBoost, two feature sets ---
XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=42, n_jobs=-1, objective="reg:squarederror")
idx_orig = [i for i in range(91) if i % 7 < 5]   # the original 5 per sensor

rows = []
print(f"\n{'XGBoost(C)':<36}{'MAE':>7}{'RMSE':>8}{'NASA':>10}")
print("-" * 62)
for cols, tag in [(idx_orig, "65 features (original)"),
                  (list(range(91)), "91 features (+quad, +ratio)")]:
    m = xgb.XGBRegressor(**XGB_PARAMS).fit(Xtr_ext[:, cols], ytr_ext)
    p = np.clip(m.predict(Xte_ext[:, cols]), 0, 125)
    mae  = mean_absolute_error(y_true, p)
    rmse = np.sqrt(mean_squared_error(y_true, p))
    ns   = nasa_score(y_true, p)
    rows.append({"features": tag, "mae": round(mae, 2),
                 "rmse": round(rmse, 2), "nasa": round(ns, 2)})
    print(f"{tag:<36}{mae:>7.2f}{rmse:>8.2f}{ns:>10.2f}")

# --- Importance share of the new features ---
imp = m.feature_importances_
names = [f"{s}_{k}" for s in FINAL_FEATURES_FD001
         for k in ["mean","std","slope","last","delta","quad","ratio"]]
new_idx = [i for i in range(91) if i % 7 >= 5]
share = 100 * sum(imp[i] for i in new_idx)
top_new = sorted([(names[i], float(imp[i])) for i in new_idx],
                 key=lambda t: -t[1])[:5]
print(f"\nNew features' share of total importance: {share:.1f}%")
print("Top new features:", [(n, round(v, 4)) for n, v in top_new])

pd.DataFrame(rows).to_csv(
    os.path.join(RESULTS_DIR, "fd001_shape_features_test.csv"), index=False)
print("\nSaved: fd001_shape_features_test.csv")

In [ ]:
# ============================================================
# NB10 — Section 8: Multi-seed check for the 91-feature XGBoost
# Confirm MAE 8.01 is stable, not a lucky seed, before adopting.
# ============================================================
import numpy as np, pandas as pd, xgboost as xgb, os
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEEDS = [42, 43, 44, 45, 46]
runs = []
for seed in SEEDS:
    params = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=seed, n_jobs=-1, objective="reg:squarederror")
    m = xgb.XGBRegressor(**params).fit(Xtr_ext, ytr_ext)   # all 91 features
    p = np.clip(m.predict(Xte_ext), 0, 125)
    mae  = mean_absolute_error(y_true, p)
    rmse = np.sqrt(mean_squared_error(y_true, p))
    ns   = nasa_score(y_true, p)
    runs.append({"seed": seed, "mae": mae, "rmse": rmse, "nasa": ns})
    print(f"  XGBoost(91) seed {seed}: MAE {mae:6.2f} | RMSE {rmse:6.2f} | NASA {ns:8.2f}")

df = pd.DataFrame(runs)
print(f"\nXGBoost (91 features) across seeds {SEEDS}:")
for m_ in ["mae", "rmse", "nasa"]:
    print(f"  {m_.upper():>5}: {df[m_].mean():8.2f} ± {df[m_].std():5.2f} "
          f"(min {df[m_].min():.2f}, max {df[m_].max():.2f})")

df.to_csv(os.path.join(RESULTS_DIR, "fd001_multiseed_xgb91.csv"), index=False)
print("\nSaved: fd001_multiseed_xgb91.csv")

In [ ]:
# ============================================================
# NB10 — Section 9: Save official per-engine predictions (91 features)
# For XGBoost (seed 42) and LR — used by Wilcoxon + final table.
# LR needs scaling (fit on train only); XGBoost does not.
# ============================================================
import numpy as np, pandas as pd, xgboost as xgb, os
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")

# --- XGBoost(91), seed 42 (matches the reported baseline) ---
XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=42, n_jobs=-1, objective="reg:squarederror")
xgbm = xgb.XGBRegressor(**XGB_PARAMS).fit(Xtr_ext, ytr_ext)
pred_xgb = np.clip(xgbm.predict(Xte_ext), 0, 125)

# --- LR(91) with scaling (fit on train only — no leakage) ---
scaler = MinMaxScaler().fit(Xtr_ext)
lr = LinearRegression().fit(scaler.transform(Xtr_ext), ytr_ext)
pred_lr = np.clip(lr.predict(scaler.transform(Xte_ext)), 0, 125)

# --- Save (SAME filenames/columns the downstream code reads) ---
pd.DataFrame({"unit": units_test, "RUL_true": y_true,
              "pred_XGB_full": pred_xgb}).to_csv(
    os.path.join(RESULTS_DIR, "fd001_xgb_full_history.csv"), index=False)
pd.DataFrame({"unit": units_test, "RUL_true": y_true,
              "pred_LR_full": pred_lr}).to_csv(
    os.path.join(RESULTS_DIR, "fd001_lr_full_history.csv"), index=False)

from sklearn.metrics import mean_absolute_error
print("Saved official 91-feature predictions:")
print(f"  XGBoost(91) seed42: MAE {mean_absolute_error(y_true, pred_xgb):.2f}")
print(f"  LR(91):             MAE {mean_absolute_error(y_true, pred_lr):.2f}")
print("  -> fd001_xgb_full_history.csv, fd001_lr_full_history.csv")

In [ ]:
# ============================================================
# NB10 — Section 10: Wilcoxon tests on the FINAL (91-feature) predictions
# All models see each engine's full history (fair).
# Reads the just-saved per-engine CSVs.
# ============================================================
import os, numpy as np, pandas as pd
from scipy.stats import wilcoxon
from itertools import combinations

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")

preds_files = {
    "LR":      ("fd001_lr_full_history.csv",      "pred_LR_full"),
    "XGBoost": ("fd001_xgb_full_history.csv",     "pred_XGB_full"),
    "LSTM":    ("fd001_lstm_full_history.csv",    "pred_LSTM_full"),
    "Chronos": ("fd001_chronos_full_history.csv", "pred_Chronos_full"),
}
abs_err = {}
for name, (fn, col) in preds_files.items():
    df = pd.read_csv(os.path.join(RESULTS_DIR, fn))
    abs_err[name] = np.abs(df[col].values - df["RUL_true"].values)

print("=== Wilcoxon signed-rank tests (FINAL, 91-feature tabular) ===")
print("    (n=100 engines, fair: all models see full history)\n")
print(f"{'Pair':<22} {'median Δ|err|':>14} {'p-value':>10} {'significant':>13}")
print("-" * 62)
rows = []
for a, b in combinations(["LR", "XGBoost", "LSTM", "Chronos"], 2):
    stat, p = wilcoxon(abs_err[a], abs_err[b])
    med = np.median(abs_err[a] - abs_err[b])
    sig = "YES" if p < 0.05 else "no"
    print(f"{a+' vs '+b:<22} {med:>+14.2f} {p:>10.4f} {sig:>13}")
    rows.append({"model_a": a, "model_b": b,
                 "median_delta_abs_err": round(med, 2),
                 "p_value": round(p, 4), "significant_5pct": sig})

pd.DataFrame(rows).to_csv(
    os.path.join(RESULTS_DIR, "fd001_wilcoxon_tests_final91.csv"), index=False)
print(f"\nSaved: fd001_wilcoxon_tests_final91.csv")

In [ ]:
# ============================================================
# NB10 — Information levels A/B/C, evaluated on validation folds
# Question: does extending the observed history improve a tabular model,
#           with the model and the target held fixed?
# Design:   GroupKFold over training engines; the group key is the engine
#           id, so no engine is split across training and validation.
#           Validation engines are truncated at a point drawn from the
#           official test RUL distribution, using the same seed as the
#           cap experiment so that all validation comparisons share the
#           same evaluation points.
# Note:     A differs from B in three respects at once — history length,
#           statistical summarisation, and dimensionality. The B-to-C
#           contrast isolates history length, since both use the same
#           five summaries per sensor.
# ============================================================
import os, time
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
RAW_PATH     = os.path.join(PROJECT_ROOT, "data", "train_FD001.txt")
RUL_PATH     = os.path.join(PROJECT_ROOT, "data", "RUL_FD001.txt")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results")

SENSORS = ["s2","s3","s4","s7","s8","s9","s11","s12","s13","s15","s17","s20","s21"]
CAP, N_FOLDS, SEED, TRUNC_SEED = 125, 5, 42, 123
WIN_B = 50

XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=SEED, n_jobs=-1, objective="reg:squarederror")

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

def summarize(w):
    """Five statistics per sensor over a window: mean, std, slope, last, delta."""
    n = w.shape[0]; x = np.arange(n); feats = []
    for j in range(w.shape[1]):
        c = w[:, j]
        slope = np.polyfit(x, c, 1)[0] if n >= 2 else 0.0
        feats.extend([c.mean(), c.std(), slope, c[-1], c[-1] - c[0]])
    return np.array(feats, dtype=np.float32)

COLS = ["unit","cycle","op1","op2","op3"] + [f"s{i}" for i in range(1,22)]
raw = pd.read_csv(RAW_PATH, sep=r"\s+", header=None, names=COLS)
raw["RUL_lin"] = raw.groupby("unit")["cycle"].transform("max") - raw["cycle"]

# --- Truncation points, identical to the cap experiment ---
test_rul = pd.read_csv(RUL_PATH, header=None).values.ravel()
rng = np.random.default_rng(TRUNC_SEED)
units = np.sort(raw["unit"].unique())
sampled = rng.choice(test_rul, size=len(units), replace=True)
trunc_rul = {u: int(v) for u, v in zip(units, sampled)}

cut_cycle = {}
for unit, g in raw.groupby("unit"):
    cand = g[g["RUL_lin"] >= trunc_rul[unit]]
    cut_cycle[unit] = int(cand["cycle"].iloc[-1]) if len(cand) else int(g["cycle"].iloc[0])

# --- Build the three representations, one sample per engine-cycle ---
# For training engines every cycle is a sample; for validation engines
# only the truncation cycle is scored, mirroring the official protocol.
t0 = time.time()
rows_A, rows_B, rows_C, y_all, unit_all, cyc_all = [], [], [], [], [], []
for unit, g in raw.groupby("unit"):
    g = g.sort_values("cycle")
    f = g[SENSORS].values.astype(np.float32)
    lab = np.minimum(g["RUL_lin"].values, CAP).astype(np.float32)
    cycles = g["cycle"].values
    for end in range(len(g)):
        hist = f[:end + 1]
        rows_A.append(f[end])                              # current cycle only
        rows_B.append(summarize(hist[-WIN_B:]))            # last 50 cycles
        rows_C.append(summarize(hist))                     # full observed history
        y_all.append(lab[end]); unit_all.append(unit); cyc_all.append(cycles[end])

reps = {
    "A_last":  np.array(rows_A, np.float32),
    "B_win50": np.array(rows_B, np.float32),
    "C_full":  np.array(rows_C, np.float32),
}
y      = np.array(y_all, np.float32)
groups = np.array(unit_all)
cycs   = np.array(cyc_all)
is_cut = np.array([cycs[i] == cut_cycle[groups[i]] for i in range(len(y))])
print(f"Built {len(y)} samples in {time.time()-t0:.0f}s | "
      f"A {reps['A_last'].shape[1]} features | "
      f"B {reps['B_win50'].shape[1]} | C {reps['C_full'].shape[1]}")
print(f"Validation points per fold: {is_cut.sum() // N_FOLDS} engines\n")

# --- Evaluate both tabular models on each representation ---
results = []
for rep_name, X in reps.items():
    for model_name in ["LR", "XGBoost"]:
        fold_mae, fold_rmse, fold_nasa = [], [], []
        for tr, va in GroupKFold(n_splits=N_FOLDS).split(X, y, groups):
            va_cut = va[is_cut[va]]
            if model_name == "LR":
                sc = MinMaxScaler().fit(X[tr])             # fit on training fold only
                m = LinearRegression().fit(sc.transform(X[tr]), y[tr])
                pred = np.clip(m.predict(sc.transform(X[va_cut])), 0, CAP)
            else:
                m = xgb.XGBRegressor(**XGB_PARAMS).fit(X[tr], y[tr])
                pred = np.clip(m.predict(X[va_cut]), 0, CAP)
            truth = y[va_cut]
            fold_mae.append(mean_absolute_error(truth, pred))
            fold_rmse.append(np.sqrt(mean_squared_error(truth, pred)))
            fold_nasa.append(nasa_score(truth, pred))
        results.append({
            "representation": rep_name, "model": model_name,
            "n_features": X.shape[1],
            "val_mae_mean": np.mean(fold_mae), "val_mae_sd": np.std(fold_mae, ddof=1),
            "val_rmse_mean": np.mean(fold_rmse),
            "val_nasa_mean": np.mean(fold_nasa),
        })
        print(f"{rep_name:<9} {model_name:<8} | "
              f"MAE {np.mean(fold_mae):6.2f} ± {np.std(fold_mae, ddof=1):4.2f} | "
              f"RMSE {np.mean(fold_rmse):6.2f} | NASA {np.mean(fold_nasa):8.2f}")

res = pd.DataFrame(results)
res.to_csv(os.path.join(RESULTS_DIR, "fd001_information_levels_validation.csv"),
           index=False)

print("\nChange in validation MAE relative to A:")
for model_name in ["LR", "XGBoost"]:
    sub = res[res.model == model_name].set_index("representation")
    base = sub.loc["A_last", "val_mae_mean"]
    for rep in ["B_win50", "C_full"]:
        v = sub.loc[rep, "val_mae_mean"]
        print(f"  {model_name:<8} {rep:<9} {v:6.2f}  ({100*(v-base)/base:+5.1f} %)")

print("\nSaved: fd001_information_levels_validation.csv")

In [ ]:
# ============================================================
# NB10 — Shape features: 65 versus 91, evaluated on validation folds
# Question: does adding a description of curvature to the summary
#           improve a tabular model, with the model, the target, and the
#           history horizon held fixed?
# Design:   identical to the A/B/C cell — same folds, same truncation
#           points, same target. The only change is the feature set.
# Causality: every feature is computed from f[:end+1], the history
#           available up to the cycle being predicted. No feature uses
#           the engine's eventual lifetime.
# Naming:   slope_ratio compares the slope over the recent third of the
#           OBSERVED history with the slope over its early third. Since
#           the observed history is truncated, its "last third" is not
#           the last third of the engine's life.
# ============================================================
import os, time
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
RAW_PATH     = os.path.join(PROJECT_ROOT, "data", "train_FD001.txt")
RUL_PATH     = os.path.join(PROJECT_ROOT, "data", "RUL_FD001.txt")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results")

SENSORS = ["s2","s3","s4","s7","s8","s9","s11","s12","s13","s15","s17","s20","s21"]
CAP, N_FOLDS, SEED, TRUNC_SEED = 125, 5, 42, 123

XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=SEED, n_jobs=-1, objective="reg:squarederror")

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

def summarize_ext(w):
    """Seven statistics per sensor: the original five plus curvature and
    a recent-to-early slope ratio. Time is normalised to [0, 1] for the
    quadratic term so that curvature is comparable across histories of
    different lengths; the linear slope stays in units per cycle."""
    n = w.shape[0]
    x_cyc  = np.arange(n)                      # per-cycle units
    x_norm = np.linspace(0.0, 1.0, n) if n > 1 else np.zeros(1)
    third = max(2, n // 3)
    feats = []
    for j in range(w.shape[1]):
        c = w[:, j]
        mean, std, last, delta = c.mean(), c.std(), c[-1], c[-1] - c[0]
        slope = np.polyfit(x_cyc, c, 1)[0] if n >= 2 else 0.0
        quad  = np.polyfit(x_norm, c, 2)[0] if n >= 3 else 0.0
        if n >= 2 * third:
            s_early  = np.polyfit(np.arange(third), c[:third],  1)[0]
            s_recent = np.polyfit(np.arange(third), c[-third:], 1)[0]
            ratio = s_recent / s_early if abs(s_early) > 1e-6 else 0.0
            ratio = float(np.clip(ratio, -50, 50))
        else:
            ratio = 0.0
        feats.extend([mean, std, slope, last, delta, quad, ratio])
    return np.array(feats, dtype=np.float32)

COLS = ["unit","cycle","op1","op2","op3"] + [f"s{i}" for i in range(1,22)]
raw = pd.read_csv(RAW_PATH, sep=r"\s+", header=None, names=COLS)
raw["RUL_lin"] = raw.groupby("unit")["cycle"].transform("max") - raw["cycle"]

# --- Truncation points, identical to the previous two experiments ---
test_rul = pd.read_csv(RUL_PATH, header=None).values.ravel()
rng = np.random.default_rng(TRUNC_SEED)
units = np.sort(raw["unit"].unique())
sampled = rng.choice(test_rul, size=len(units), replace=True)
trunc_rul = {u: int(v) for u, v in zip(units, sampled)}

cut_cycle = {}
for unit, g in raw.groupby("unit"):
    cand = g[g["RUL_lin"] >= trunc_rul[unit]]
    cut_cycle[unit] = int(cand["cycle"].iloc[-1]) if len(cand) else int(g["cycle"].iloc[0])

# --- Build the 91-feature representation over the full observed history ---
t0 = time.time()
X91, y_all, unit_all, cyc_all = [], [], [], []
for unit, g in raw.groupby("unit"):
    g = g.sort_values("cycle")
    f = g[SENSORS].values.astype(np.float32)
    lab = np.minimum(g["RUL_lin"].values, CAP).astype(np.float32)
    cycles = g["cycle"].values
    for end in range(len(g)):
        X91.append(summarize_ext(f[:end + 1]))
        y_all.append(lab[end]); unit_all.append(unit); cyc_all.append(cycles[end])

X91 = np.array(X91, np.float32)
y      = np.array(y_all, np.float32)
groups = np.array(unit_all)
cycs   = np.array(cyc_all)
is_cut = np.array([cycs[i] == cut_cycle[groups[i]] for i in range(len(y))])

# The original five statistics occupy positions 0-4 of each block of seven,
# so the 65-feature set is a strict subset of the 91-feature set. Both are
# therefore evaluated on identical rows.
idx65 = [i for i in range(X91.shape[1]) if i % 7 < 5]
sets = {"65_base": X91[:, idx65], "91_shape": X91}

print(f"Built {len(y)} samples in {time.time()-t0:.0f}s | "
      f"65-feature and 91-feature sets share the same rows")
print(f"Validation points per fold: {is_cut.sum() // N_FOLDS} engines\n")

# --- Evaluate ---
results = []
for set_name, X in sets.items():
    for model_name in ["LR", "XGBoost"]:
        fold_mae, fold_rmse, fold_nasa = [], [], []
        for tr, va in GroupKFold(n_splits=N_FOLDS).split(X, y, groups):
            va_cut = va[is_cut[va]]
            if model_name == "LR":
                sc = MinMaxScaler().fit(X[tr])
                m = LinearRegression().fit(sc.transform(X[tr]), y[tr])
                pred = np.clip(m.predict(sc.transform(X[va_cut])), 0, CAP)
            else:
                m = xgb.XGBRegressor(**XGB_PARAMS).fit(X[tr], y[tr])
                pred = np.clip(m.predict(X[va_cut]), 0, CAP)
            truth = y[va_cut]
            fold_mae.append(mean_absolute_error(truth, pred))
            fold_rmse.append(np.sqrt(mean_squared_error(truth, pred)))
            fold_nasa.append(nasa_score(truth, pred))
        results.append({
            "feature_set": set_name, "model": model_name,
            "n_features": X.shape[1],
            "val_mae_mean": np.mean(fold_mae), "val_mae_sd": np.std(fold_mae, ddof=1),
            "val_rmse_mean": np.mean(fold_rmse),
            "val_nasa_mean": np.mean(fold_nasa),
        })
        print(f"{set_name:<9} {model_name:<8} ({X.shape[1]:>2} feat) | "
              f"MAE {np.mean(fold_mae):6.2f} ± {np.std(fold_mae, ddof=1):4.2f} | "
              f"RMSE {np.mean(fold_rmse):6.2f} | NASA {np.mean(fold_nasa):8.2f}")

res = pd.DataFrame(results)
res.to_csv(os.path.join(RESULTS_DIR, "fd001_shape_features_validation.csv"),
           index=False)

print("\nEffect of adding the shape features:")
for model_name in ["LR", "XGBoost"]:
    sub = res[res.model == model_name].set_index("feature_set")
    b, s = sub.loc["65_base", "val_mae_mean"], sub.loc["91_shape", "val_mae_mean"]
    sd = sub.loc["91_shape", "val_mae_sd"]
    verdict = "smaller than the fold-to-fold SD" if abs(b - s) < sd else "larger than the fold-to-fold SD"
    print(f"  {model_name:<8} {b:6.2f} -> {s:6.2f}  ({100*(s-b)/b:+5.1f} %), "
          f"difference {verdict} ({sd:.2f})")

print("\nSaved: fd001_shape_features_validation.csv")

In [ ]:
# ============================================================
# NB10 — Official per-engine predictions, 65-feature representation
# The shape features were tested on validation folds and produced no
# measurable improvement, so the simpler 65-feature summary of the full
# observed history is adopted as the tabular representation.
# Both train and test are read from the processed files, so the two are
# on the same scale.
# ============================================================
import os, time
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")

import sys
sys.path.insert(0, SRC_DIR)
from preprocessing import FINAL_FEATURES_FD001

CAP, SEED = 125, 42
XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=SEED, n_jobs=-1, objective="reg:squarederror")

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

def summarize(w):
    """Five statistics per sensor: mean, std, slope, last, delta."""
    n = w.shape[0]; x = np.arange(n); feats = []
    for j in range(w.shape[1]):
        c = w[:, j]
        slope = np.polyfit(x, c, 1)[0] if n >= 2 else 0.0
        feats.extend([c.mean(), c.std(), slope, c[-1], c[-1] - c[0]])
    return np.array(feats, dtype=np.float32)

# --- Both sets come from the processed files, so they share one scale ---
train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))

units_test = test_last["unit"].values
y_true = test_last["RUL_true"].values
print(f"Sensors: {len(FINAL_FEATURES_FD001)} | train {train.shape} | test {test_full.shape}")

t0 = time.time()
Xtr, ytr = [], []
for unit, g in train.groupby("unit"):
    g = g.sort_values("cycle")
    f = g[FINAL_FEATURES_FD001].values.astype(np.float32)
    lab = g["RUL_piecewise125"].values.astype(np.float32)
    for end in range(len(g)):
        Xtr.append(summarize(f[:end + 1]))
        ytr.append(lab[end])
Xtr, ytr = np.array(Xtr, np.float32), np.array(ytr, np.float32)

Xte = np.array([
    summarize(test_full[test_full["unit"] == u]
              .sort_values("cycle")[FINAL_FEATURES_FD001].values.astype(np.float32))
    for u in units_test
], np.float32)
print(f"Built train {Xtr.shape} and test {Xte.shape} in {time.time()-t0:.0f}s")

# --- Fit and predict ---
scaler = MinMaxScaler().fit(Xtr)                      # fit on training data only
lr = LinearRegression().fit(scaler.transform(Xtr), ytr)
pred_lr = np.clip(lr.predict(scaler.transform(Xte)), 0, CAP)

xgbm = xgb.XGBRegressor(**XGB_PARAMS).fit(Xtr, ytr)
pred_xgb = np.clip(xgbm.predict(Xte), 0, CAP)

# --- Save under the filenames the downstream cells read ---
pd.DataFrame({"unit": units_test, "RUL_true": y_true,
              "pred_LR_full": pred_lr}).to_csv(
    os.path.join(RESULTS_DIR, "fd001_lr_full_history.csv"), index=False)
pd.DataFrame({"unit": units_test, "RUL_true": y_true,
              "pred_XGB_full": pred_xgb}).to_csv(
    os.path.join(RESULTS_DIR, "fd001_xgb_full_history.csv"), index=False)

for name, p in [("LR", pred_lr), ("XGBoost", pred_xgb)]:
    print(f"{name:<8} MAE {mean_absolute_error(y_true, p):6.2f} | "
          f"RMSE {np.sqrt(mean_squared_error(y_true, p)):6.2f} | "
          f"NASA {nasa_score(y_true, p):8.2f}")

print("\nSaved: fd001_lr_full_history.csv, fd001_xgb_full_history.csv")

*italicized text*

**jj**

In [ ]:
# ============================================================
# NB10 — Multi-seed check for the 65-feature XGBoost
# XGBoost draws random subsamples of rows and columns at each tree, so
# the fitted model depends on the seed. Repeating the fit across seeds
# establishes whether the reported error is stable.
# ============================================================
import numpy as np, pandas as pd, xgboost as xgb, os
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEEDS = [42, 43, 44, 45, 46]
runs = []
for seed in SEEDS:
    params = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=seed, n_jobs=-1, objective="reg:squarederror")
    m = xgb.XGBRegressor(**params).fit(Xtr, ytr)
    p = np.clip(m.predict(Xte), 0, 125)
    mae  = mean_absolute_error(y_true, p)
    rmse = np.sqrt(mean_squared_error(y_true, p))
    ns   = nasa_score(y_true, p)
    runs.append({"seed": seed, "mae": mae, "rmse": rmse, "nasa": ns})
    print(f"  seed {seed}: MAE {mae:6.2f} | RMSE {rmse:6.2f} | NASA {ns:8.2f}")

df = pd.DataFrame(runs)
print(f"\nXGBoost (65 features) across {len(SEEDS)} seeds:")
for col, label in [("mae", "MAE"), ("rmse", "RMSE"), ("nasa", "NASA")]:
    print(f"  {label:>5}: {df[col].mean():8.2f} ± {df[col].std():5.2f} "
          f"(min {df[col].min():.2f}, max {df[col].max():.2f})")

df.to_csv(os.path.join(RESULTS_DIR, "fd001_multiseed_xgb65.csv"), index=False)
print("\nSaved: fd001_multiseed_xgb65.csv")


In [ ]:
# ============================================================
# NB10 — Multi-seed check for the 65-feature XGBoost
# XGBoost draws random subsamples of rows and columns at each tree, so
# the fitted model depends on the seed. Repeating the fit across seeds
# establishes whether the reported error is stable.
# ============================================================
import numpy as np, pandas as pd, xgboost as xgb, os
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEEDS = [42, 43, 44, 45, 46]
runs = []
for seed in SEEDS:
    params = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=seed, n_jobs=-1, objective="reg:squarederror")
    m = xgb.XGBRegressor(**params).fit(Xtr, ytr)
    p = np.clip(m.predict(Xte), 0, 125)
    mae  = mean_absolute_error(y_true, p)
    rmse = np.sqrt(mean_squared_error(y_true, p))
    ns   = nasa_score(y_true, p)
    runs.append({"seed": seed, "mae": mae, "rmse": rmse, "nasa": ns})
    print(f"  seed {seed}: MAE {mae:6.2f} | RMSE {rmse:6.2f} | NASA {ns:8.2f}")

df = pd.DataFrame(runs)
print(f"\nXGBoost (65 features) across {len(SEEDS)} seeds:")
for col, label in [("mae", "MAE"), ("rmse", "RMSE"), ("nasa", "NASA")]:
    print(f"  {label:>5}: {df[col].mean():8.2f} ± {df[col].std():5.2f} "
          f"(min {df[col].min():.2f}, max {df[col].max():.2f})")

df.to_csv(os.path.join(RESULTS_DIR, "fd001_multiseed_xgb65.csv"), index=False)
print("\nSaved: fd001_multiseed_xgb65.csv")


In [ ]:
# ============================================================
# NB10 — Multi-seed XGBoost, saving per-seed predictions
# Change from the earlier version: the prediction of every seed is now
# written to disk, not only that of the first. The significance analysis
# needs one error per engine averaged across seeds, which matches how
# the reported mean is formed; using a single seed's predictions there
# while reporting a five-seed mean would compare two different things.
# Self-contained: rebuilds the feature matrices from the processed files.
# ============================================================
import os, sys, time
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
PRED_DIR      = os.path.join(RESULTS_DIR, "per_seed")
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
from preprocessing import FINAL_FEATURES_FD001

os.makedirs(PRED_DIR, exist_ok=True)
CAP, SEEDS = 125, [42, 43, 44, 45, 46]

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

def summarize(w):
    """Five statistics per sensor: mean, std, slope, last, delta."""
    n = w.shape[0]; x = np.arange(n); feats = []
    for j in range(w.shape[1]):
        c = w[:, j]
        slope = np.polyfit(x, c, 1)[0] if n >= 2 else 0.0
        feats.extend([c.mean(), c.std(), slope, c[-1], c[-1] - c[0]])
    return np.array(feats, dtype=np.float32)

train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
units_test = test_last["unit"].values
y_true = test_last["RUL_true"].values

t0 = time.time()
Xtr, ytr = [], []
for unit, g in train.groupby("unit"):
    g = g.sort_values("cycle")
    f = g[FINAL_FEATURES_FD001].values.astype(np.float32)
    lab = g["RUL_piecewise125"].values.astype(np.float32)
    for end in range(len(g)):
        Xtr.append(summarize(f[:end + 1])); ytr.append(lab[end])
Xtr, ytr = np.array(Xtr, np.float32), np.array(ytr, np.float32)

Xte = np.array([
    summarize(test_full[test_full["unit"] == u]
              .sort_values("cycle")[FINAL_FEATURES_FD001].values.astype(np.float32))
    for u in units_test
], np.float32)
print(f"Built train {Xtr.shape} and test {Xte.shape} in {time.time()-t0:.0f}s\n")

runs, preds = [], {}
for seed in SEEDS:
    params = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=seed, n_jobs=-1, objective="reg:squarederror")
    t0 = time.time()
    m = xgb.XGBRegressor(**params).fit(Xtr, ytr)
    t_train = time.time() - t0
    p = np.clip(m.predict(Xte), 0, CAP)
    preds[seed] = p
    runs.append({"seed": seed,
                 "mae": mean_absolute_error(y_true, p),
                 "rmse": np.sqrt(mean_squared_error(y_true, p)),
                 "nasa": nasa_score(y_true, p),
                 "train_s": t_train})
    print(f"  seed {seed}: MAE {runs[-1]['mae']:6.2f} | RMSE {runs[-1]['rmse']:6.2f} | "
          f"NASA {runs[-1]['nasa']:8.2f} | {t_train:.1f}s")

df = pd.DataFrame(runs)
print(f"\nXGBoost (65 features) across {len(SEEDS)} seeds:")
for m_ in ["mae", "rmse", "nasa"]:
    print(f"  {m_.upper():>5}: {df[m_].mean():8.2f} ± {df[m_].std():5.2f} "
          f"(min {df[m_].min():.2f}, max {df[m_].max():.2f})")
print(f"  Mean training time: {df['train_s'].mean():.1f}s on CPU")

df["model"] = "XGBoost"
df.to_csv(os.path.join(RESULTS_DIR, "fd001_multiseed_xgb65.csv"), index=False)

# Per-seed predictions, one column per seed
wide = pd.DataFrame({"unit": units_test, "RUL_true": y_true})
for seed in SEEDS:
    wide[f"pred_seed{seed}"] = preds[seed]
wide.to_csv(os.path.join(PRED_DIR, "xgb_per_seed.csv"), index=False)

# Keep the single-seed file the earlier cells read, for continuity
pd.DataFrame({"unit": units_test, "RUL_true": y_true,
              "pred_XGB_full": preds[SEEDS[0]]}).to_csv(
    os.path.join(RESULTS_DIR, "fd001_xgb_full_history.csv"), index=False)

print(f"\nSaved: fd001_multiseed_xgb65.csv + per_seed/xgb_per_seed.csv")

# ============================================================
# NB10 — Wilcoxon signed-rank tests on the final predictions
# The four models are compared pairwise on their per-engine absolute
# errors. The test is two-sided and paired: each engine contributes one
# error from each model, so the comparison is within-engine. Holm
# correction is applied across the six pairwise tests.
# ============================================================
import os, numpy as np, pandas as pd
from scipy.stats import wilcoxon
from itertools import combinations
from statsmodels.stats.multitest import multipletests

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")

preds = {
    "LR":      ("fd001_lr_full_history.csv",      "pred_LR_full"),
    "XGBoost": ("fd001_xgb_full_history.csv",     "pred_XGB_full"),
    "LSTM":    ("fd001_lstm_full_history.csv",    "pred_LSTM_full"),
    "Chronos": ("fd001_chronos_full_history.csv", "pred_Chronos_full"),
}

abs_err, units_ref = {}, None
for name, (fn, col) in preds.items():
    df = pd.read_csv(os.path.join(RESULTS_DIR, fn)).sort_values("unit")
    if units_ref is None:
        units_ref = df["unit"].values
    assert np.array_equal(df["unit"].values, units_ref), f"{name}: engine order differs"
    abs_err[name] = np.abs(df[col].values - df["RUL_true"].values)

print(f"Paired over {len(units_ref)} engines, 65-feature tabular representation\n")

rows = []
for a, b in combinations(preds.keys(), 2):
    d = abs_err[a] - abs_err[b]
    n_zero = int((d == 0).sum())
    stat, p = wilcoxon(abs_err[a], abs_err[b], alternative="two-sided",
                       zero_method="wilcox")
    rows.append({"model_a": a, "model_b": b,
                 "median_diff": float(np.median(d)),
                 "n_zero_diff": n_zero, "p_raw": p})

res = pd.DataFrame(rows)
res["p_holm"] = multipletests(res["p_raw"], method="holm")[1]
res["significant_5pct"] = np.where(res["p_holm"] < 0.05, "yes", "no")

def fmt(p):
    return "<0.0001" if p < 1e-4 else f"{p:.4f}"

print(f"{'Pair':<22}{'median diff':>13}{'p (raw)':>10}{'p (Holm)':>10}{'sig.':>7}")
print("-" * 62)
for _, r in res.iterrows():
    print(f"{r.model_a + ' vs ' + r.model_b:<22}{r.median_diff:>+13.2f}"
          f"{fmt(r.p_raw):>10}{fmt(r.p_holm):>10}{r.significant_5pct:>7}")

res.to_csv(os.path.join(RESULTS_DIR, "fd001_wilcoxon_final65.csv"), index=False)
print(f"\nZero differences per pair: {res['n_zero_diff'].tolist()}")
print("Saved: fd001_wilcoxon_final65.csv")

In [ ]:
# ============================================================
# NB10 — Wilcoxon signed-rank tests on the final predictions
# Change from the earlier version: for the three stochastic models the
# absolute error is computed for every seed and engine, then averaged
# across seeds. This gives one error per model and engine and matches
# how the reported means are formed; comparing a single seed's
# predictions against a five-seed mean would test a different quantity.
# The 500 seed-engine combinations are not treated as independent.
# ============================================================
import os
import numpy as np, pandas as pd
from scipy.stats import wilcoxon
from itertools import combinations
from statsmodels.stats.multitest import multipletests

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")
PRED_DIR    = os.path.join(RESULTS_DIR, "per_seed")

# LR is deterministic and has a single prediction file; the other three
# have one column per seed.
SINGLE = {"LR": ("fd001_lr_full_history.csv", "pred_LR_full")}
MULTI  = {"XGBoost": "xgb_per_seed.csv",
          "LSTM":    "lstm_per_seed.csv",
          "Chronos": "chronos_per_seed.csv"}

abs_err, units_ref = {}, None

for name, (fn, col) in SINGLE.items():
    df = pd.read_csv(os.path.join(RESULTS_DIR, fn)).sort_values("unit")
    units_ref = df["unit"].values if units_ref is None else units_ref
    assert np.array_equal(df["unit"].values, units_ref), f"{name}: engine order differs"
    abs_err[name] = np.abs(df[col].values - df["RUL_true"].values)

for name, fn in MULTI.items():
    df = pd.read_csv(os.path.join(PRED_DIR, fn)).sort_values("unit")
    assert np.array_equal(df["unit"].values, units_ref), f"{name}: engine order differs"
    seed_cols = [c for c in df.columns if c.startswith("pred_seed")]
    errs = np.abs(df[seed_cols].values - df["RUL_true"].values[:, None])
    abs_err[name] = errs.mean(axis=1)          # mean over seeds, per engine
    print(f"{name:<9} {len(seed_cols)} seeds | mean of per-engine mean error: "
          f"{abs_err[name].mean():.2f}")

print(f"\nPaired over {len(units_ref)} engines\n")

order = ["LR", "XGBoost", "LSTM", "Chronos"]
rows = []
for a, b in combinations(order, 2):
    d = abs_err[a] - abs_err[b]
    stat, p = wilcoxon(abs_err[a], abs_err[b], alternative="two-sided",
                       zero_method="wilcox")
    rows.append({"model_a": a, "model_b": b,
                 "median_diff": float(np.median(d)),
                 "n_zero_diff": int((d == 0).sum()),
                 "p_raw": p})

res = pd.DataFrame(rows)
res["p_holm"] = multipletests(res["p_raw"], method="holm")[1]
res["significant_5pct"] = np.where(res["p_holm"] < 0.05, "yes", "no")

fmt = lambda p: "<0.0001" if p < 1e-4 else f"{p:.4f}"
print(f"{'Pair':<22}{'median diff':>13}{'p (raw)':>10}{'p (Holm)':>10}{'sig.':>7}")
print("-" * 62)
for _, r in res.iterrows():
    print(f"{r.model_a + ' vs ' + r.model_b:<22}{r.median_diff:>+13.2f}"
          f"{fmt(r.p_raw):>10}{fmt(r.p_holm):>10}{r.significant_5pct:>7}")

res.to_csv(os.path.join(RESULTS_DIR, "fd001_wilcoxon_final65.csv"), index=False)
print(f"\nZero differences per pair: {res['n_zero_diff'].tolist()}")
print("Saved: fd001_wilcoxon_final65.csv")

In [ ]:
# ============================================================
# NB10 — Smoothing, evaluated on validation folds
# Question: does smoothing the sensor signals before summarising them
#           improve a tabular model?
# Design:   identical to the A/B/C and shape-feature cells — same folds,
#           same truncation points, same target, same 65-feature summary.
#           The only change is whether the signal is smoothed first.
# Causality: the moving average is trailing, so the value at cycle t uses
#           only cycles up to t. A centred window would use future
#           cycles and could not be computed in operation.
# ============================================================
import os, time
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RUL_PATH      = os.path.join(PROJECT_ROOT, "data", "RUL_FD001.txt")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")

import sys
sys.path.insert(0, SRC_DIR)
from preprocessing import FINAL_FEATURES_FD001

CAP, N_FOLDS, SEED, TRUNC_SEED = 125, 5, 42, 123
WINDOWS = [1, 5, 10, 20]          # 1 = no smoothing

XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=SEED, n_jobs=-1, objective="reg:squarederror")

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

def summarize(w):
    """Five statistics per sensor: mean, std, slope, last, delta."""
    n = w.shape[0]; x = np.arange(n); feats = []
    for j in range(w.shape[1]):
        c = w[:, j]
        slope = np.polyfit(x, c, 1)[0] if n >= 2 else 0.0
        feats.extend([c.mean(), c.std(), slope, c[-1], c[-1] - c[0]])
    return np.array(feats, dtype=np.float32)

def smooth(f, window):
    """Trailing moving average. min_periods=1 keeps the early cycles
    without introducing NaNs and without using any future value."""
    if window <= 1:
        return f
    return (pd.DataFrame(f).rolling(window=window, min_periods=1)
            .mean().values.astype(np.float32))

train = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
train["RUL_lin"] = train.groupby("unit")["cycle"].transform("max") - train["cycle"]

# --- Truncation points, identical to the earlier validation experiments ---
test_rul = pd.read_csv(RUL_PATH, header=None).values.ravel()
rng = np.random.default_rng(TRUNC_SEED)
units = np.sort(train["unit"].unique())
sampled = rng.choice(test_rul, size=len(units), replace=True)
trunc_rul = {u: int(v) for u, v in zip(units, sampled)}

cut_cycle = {}
for unit, g in train.groupby("unit"):
    cand = g[g["RUL_lin"] >= trunc_rul[unit]]
    cut_cycle[unit] = int(cand["cycle"].iloc[-1]) if len(cand) else int(g["cycle"].iloc[0])

# --- Build one feature matrix per smoothing window ---
sets, y_all, unit_all, cyc_all = {}, None, None, None
for win in WINDOWS:
    t0 = time.time()
    X, y_tmp, u_tmp, c_tmp = [], [], [], []
    for unit, g in train.groupby("unit"):
        g = g.sort_values("cycle")
        f = smooth(g[FINAL_FEATURES_FD001].values.astype(np.float32), win)
        lab = g["RUL_piecewise125"].values.astype(np.float32)
        cycles = g["cycle"].values
        for end in range(len(g)):
            X.append(summarize(f[:end + 1]))
            y_tmp.append(lab[end]); u_tmp.append(unit); c_tmp.append(cycles[end])
    sets[win] = np.array(X, np.float32)
    if y_all is None:
        y_all, unit_all, cyc_all = (np.array(y_tmp, np.float32),
                                    np.array(u_tmp), np.array(c_tmp))
    label = "no smoothing" if win == 1 else f"window {win}"
    print(f"Built {label:<14} in {time.time()-t0:>3.0f}s")

y, groups, cycs = y_all, unit_all, cyc_all
is_cut = np.array([cycs[i] == cut_cycle[groups[i]] for i in range(len(y))])
print(f"\n{len(y)} samples | validation points per fold: {is_cut.sum() // N_FOLDS} engines\n")

# --- Evaluate ---
results = []
for win, X in sets.items():
    for model_name in ["LR", "XGBoost"]:
        fold_mae, fold_rmse, fold_nasa = [], [], []
        for tr, va in GroupKFold(n_splits=N_FOLDS).split(X, y, groups):
            va_cut = va[is_cut[va]]
            if model_name == "LR":
                sc = MinMaxScaler().fit(X[tr])
                m = LinearRegression().fit(sc.transform(X[tr]), y[tr])
                pred = np.clip(m.predict(sc.transform(X[va_cut])), 0, CAP)
            else:
                m = xgb.XGBRegressor(**XGB_PARAMS).fit(X[tr], y[tr])
                pred = np.clip(m.predict(X[va_cut]), 0, CAP)
            truth = y[va_cut]
            fold_mae.append(mean_absolute_error(truth, pred))
            fold_rmse.append(np.sqrt(mean_squared_error(truth, pred)))
            fold_nasa.append(nasa_score(truth, pred))
        results.append({"smoothing_window": win, "model": model_name,
                        "val_mae_mean": np.mean(fold_mae),
                        "val_mae_sd": np.std(fold_mae, ddof=1),
                        "val_rmse_mean": np.mean(fold_rmse),
                        "val_nasa_mean": np.mean(fold_nasa)})
        label = "none" if win == 1 else str(win)
        print(f"window {label:<5} {model_name:<8} | "
              f"MAE {np.mean(fold_mae):6.2f} ± {np.std(fold_mae, ddof=1):4.2f} | "
              f"RMSE {np.mean(fold_rmse):6.2f} | NASA {np.mean(fold_nasa):8.2f}")

res = pd.DataFrame(results)
res.to_csv(os.path.join(RESULTS_DIR, "fd001_smoothing_validation.csv"), index=False)

print("\nChange relative to no smoothing:")
for model_name in ["LR", "XGBoost"]:
    sub = res[res.model == model_name].set_index("smoothing_window")
    base = sub.loc[1, "val_mae_mean"]
    for win in WINDOWS[1:]:
        v, sd = sub.loc[win, "val_mae_mean"], sub.loc[win, "val_mae_sd"]
        verdict = "within" if abs(v - base) < sd else "beyond"
        print(f"  {model_name:<8} window {win:>2}: {base:6.2f} -> {v:6.2f} "
              f"({100*(v-base)/base:+5.1f} %), {verdict} the fold-to-fold SD ({sd:.2f})")

print("\nSaved: fd001_smoothing_validation.csv")

In [ ]:
# ============================================================
# NB10 — Effect of the target definition, on the final representation
# Both tabular models are refitted on the linear target, using the same
# 65-feature summary of the full observed history as the main results.
# An earlier comparison existed but used the single-cycle representation
# and reported only the PHM score, so it is not comparable with the
# figures reported elsewhere.
# Clipping differs by definition: predictions on the piecewise target are
# bounded at 125, whereas bounding a linear-target prediction there would
# convert its output to the piecewise form at evaluation.
# ============================================================
import os, sys, time
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
from preprocessing import FINAL_FEATURES_FD001

CAP, SEED = 125, 42
XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=SEED, n_jobs=-1, objective="reg:squarederror")

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

def summarize(w):
    n = w.shape[0]; x = np.arange(n); feats = []
    for j in range(w.shape[1]):
        c = w[:, j]
        slope = np.polyfit(x, c, 1)[0] if n >= 2 else 0.0
        feats.extend([c.mean(), c.std(), slope, c[-1], c[-1] - c[0]])
    return np.array(feats, dtype=np.float32)

train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
train["RUL_lin"] = train.groupby("unit")["cycle"].transform("max") - train["cycle"]
units_test = test_last["unit"].values
y_true = test_last["RUL_true"].values

t0 = time.time()
Xtr, y_lin, y_pw = [], [], []
for unit, g in train.groupby("unit"):
    g = g.sort_values("cycle")
    f = g[FINAL_FEATURES_FD001].values.astype(np.float32)
    lin = g["RUL_lin"].values.astype(np.float32)
    pw  = g["RUL_piecewise125"].values.astype(np.float32)
    for end in range(len(g)):
        Xtr.append(summarize(f[:end + 1])); y_lin.append(lin[end]); y_pw.append(pw[end])
Xtr = np.array(Xtr, np.float32)
y_lin, y_pw = np.array(y_lin, np.float32), np.array(y_pw, np.float32)

Xte = np.array([
    summarize(test_full[test_full["unit"] == u]
              .sort_values("cycle")[FINAL_FEATURES_FD001].values.astype(np.float32))
    for u in units_test
], np.float32)
print(f"Built train {Xtr.shape} and test {Xte.shape} in {time.time()-t0:.0f}s")
print(f"Linear target: max {y_lin.max():.0f} | piecewise target: max {y_pw.max():.0f}\n")

rows = []
for label, y_tr_, upper in [("linear", y_lin, None), ("piecewise", y_pw, CAP)]:
    # Linear regression
    sc = MinMaxScaler().fit(Xtr)
    lr = LinearRegression().fit(sc.transform(Xtr), y_tr_)
    p = lr.predict(sc.transform(Xte))
    p = np.clip(p, 0, upper) if upper else np.maximum(p, 0)
    rows.append({"model": "LR", "target": label,
                 "mae": mean_absolute_error(y_true, p),
                 "rmse": np.sqrt(mean_squared_error(y_true, p)),
                 "nasa": nasa_score(y_true, p)})

    # XGBoost
    m = xgb.XGBRegressor(**XGB_PARAMS).fit(Xtr, y_tr_)
    p = m.predict(Xte)
    p = np.clip(p, 0, upper) if upper else np.maximum(p, 0)
    rows.append({"model": "XGBoost", "target": label,
                 "mae": mean_absolute_error(y_true, p),
                 "rmse": np.sqrt(mean_squared_error(y_true, p)),
                 "nasa": nasa_score(y_true, p)})

res = pd.DataFrame(rows)
print(f"{'Model':<10}{'Target':<12}{'MAE':>8}{'RMSE':>8}{'NASA':>12}")
print("-" * 50)
for _, r in res.iterrows():
    print(f"{r.model:<10}{r.target:<12}{r.mae:>8.2f}{r.rmse:>8.2f}{r.nasa:>12.2f}")

print("\nChange from linear to piecewise:")
for mdl in ["LR", "XGBoost"]:
    s = res[res.model == mdl].set_index("target")
    for met in ["mae", "rmse", "nasa"]:
        a, b = s.loc["linear", met], s.loc["piecewise", met]
        print(f"  {mdl:<8} {met.upper():>5}: {a:8.2f} -> {b:8.2f}  ({100*(b-a)/a:+6.1f} %)")

res.to_csv(os.path.join(RESULTS_DIR, "fd001_target_definition_final.csv"), index=False)
print("\nSaved: fd001_target_definition_final.csv")

In [ ]:
# ============================================================
# NB10 — Permutation importance for the final XGBoost configuration
# Computed on the 65-feature representation actually adopted, not on the
# 91-feature version that was tested and rejected.
# Permutation importance is used rather than the built-in split counts:
# it measures the rise in error when one input is shuffled, so it
# reflects what the fitted model relies on when predicting, and is
# comparable across features of different scales.
# ============================================================
import os, sys, time
import numpy as np, pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
FIGURES_DIR   = os.path.join(PROJECT_ROOT, "figures")
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
from preprocessing import FINAL_FEATURES_FD001

CAP, SEED, N_REPEATS = 125, 42, 10
STATS = ["mean", "std", "slope", "last", "delta"]
XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=SEED, n_jobs=-1, objective="reg:squarederror")

def summarize(w):
    n = w.shape[0]; x = np.arange(n); feats = []
    for j in range(w.shape[1]):
        c = w[:, j]
        slope = np.polyfit(x, c, 1)[0] if n >= 2 else 0.0
        feats.extend([c.mean(), c.std(), slope, c[-1], c[-1] - c[0]])
    return np.array(feats, dtype=np.float32)

# Column names follow the order in which summarize emits them
feat_names = [f"{s}_{st}" for s in FINAL_FEATURES_FD001 for st in STATS]

train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
units_test = test_last["unit"].values
y_true = test_last["RUL_true"].values

t0 = time.time()
Xtr, ytr = [], []
for unit, g in train.groupby("unit"):
    g = g.sort_values("cycle")
    f = g[FINAL_FEATURES_FD001].values.astype(np.float32)
    lab = g["RUL_piecewise125"].values.astype(np.float32)
    for end in range(len(g)):
        Xtr.append(summarize(f[:end + 1])); ytr.append(lab[end])
Xtr, ytr = np.array(Xtr, np.float32), np.array(ytr, np.float32)

Xte = np.array([
    summarize(test_full[test_full["unit"] == u]
              .sort_values("cycle")[FINAL_FEATURES_FD001].values.astype(np.float32))
    for u in units_test
], np.float32)
print(f"Built train {Xtr.shape} and test {Xte.shape} in {time.time()-t0:.0f}s | "
      f"{len(feat_names)} feature names")

model = xgb.XGBRegressor(**XGB_PARAMS).fit(Xtr, ytr)
base = mean_absolute_error(y_true, np.clip(model.predict(Xte), 0, CAP))
print(f"Baseline test MAE: {base:.2f}\n")

r = permutation_importance(model, Xte, y_true, scoring="neg_mean_absolute_error",
                           n_repeats=N_REPEATS, random_state=SEED, n_jobs=-1)
imp = pd.DataFrame({"feature": feat_names,
                    "importance": r.importances_mean,
                    "std": r.importances_std})
imp["sensor"]    = [f.rsplit("_", 1)[0] for f in imp["feature"]]
imp["statistic"] = [f.rsplit("_", 1)[1] for f in imp["feature"]]
imp = imp.sort_values("importance", ascending=False).reset_index(drop=True)

print("Top ten features (rise in MAE when the feature is shuffled):")
for _, row in imp.head(10).iterrows():
    print(f"  {row.feature:<14} {row.importance:6.3f} ± {row['std']:5.3f}")

by_stat = imp.groupby("statistic")["importance"].sum().sort_values(ascending=False)
print("\nBy summary statistic (sum over the thirteen sensors):")
for k, v in by_stat.items():
    print(f"  {k:<8} {v:7.3f}")

by_sensor = imp.groupby("sensor")["importance"].sum().sort_values(ascending=False)
print("\nTop five sensors (sum over the five statistics):")
for k, v in by_sensor.head(5).items():
    print(f"  {k:<6} {v:7.3f}")

imp.to_csv(os.path.join(RESULTS_DIR, "fd001_xgb_importance_65.csv"), index=False)

# --- Figure: top fifteen features, and the totals by statistic ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5),
                               gridspec_kw={"width_ratios": [1.7, 1]})

top = imp.head(15).iloc[::-1]
ax1.barh(top["feature"], top["importance"], xerr=top["std"],
         color="#2E75B6", edgecolor="black", linewidth=0.6, capsize=3)
ax1.set_xlabel("Increase in MAE when the feature is shuffled (cycles)")
ax1.grid(True, axis="x", alpha=0.3); ax1.set_axisbelow(True)

ax2.barh(by_stat.index[::-1], by_stat.values[::-1],
         color="#C55A11", edgecolor="black", linewidth=0.6)
ax2.set_xlabel("Summed increase in MAE (cycles)")
ax2.grid(True, axis="x", alpha=0.3); ax2.set_axisbelow(True)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "fd001_xgb_importance_65.png"),
            dpi=300, bbox_inches="tight")
plt.show()
print("\nSaved: fd001_xgb_importance_65.csv + fd001_xgb_importance_65.png")

In [ ]:
# ============================================================
# NB10 — Effect of the target definition, five seeds
# Change from the earlier version: the comparison now uses the same
# five-seed protocol as the main results, so the piecewise figures here
# match those reported in the final comparison rather than differing by
# the seed. Linear regression is deterministic and is fitted once.
# Both definitions are evaluated against the same unrestricted official
# test RUL; only the output constraint differs, since bounding a
# linear-target prediction at 125 would convert it to the piecewise form.
# ============================================================
import os, sys, time
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
from preprocessing import FINAL_FEATURES_FD001

CAP, SEEDS = 125, [42, 43, 44, 45, 46]

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

def summarize(w):
    n = w.shape[0]; x = np.arange(n); feats = []
    for j in range(w.shape[1]):
        c = w[:, j]
        slope = np.polyfit(x, c, 1)[0] if n >= 2 else 0.0
        feats.extend([c.mean(), c.std(), slope, c[-1], c[-1] - c[0]])
    return np.array(feats, dtype=np.float32)

train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
train["RUL_lin"] = train.groupby("unit")["cycle"].transform("max") - train["cycle"]
units_test = test_last["unit"].values
y_true = test_last["RUL_true"].values          # unrestricted, the same for both

t0 = time.time()
Xtr, y_lin, y_pw = [], [], []
for unit, g in train.groupby("unit"):
    g = g.sort_values("cycle")
    f = g[FINAL_FEATURES_FD001].values.astype(np.float32)
    lin = g["RUL_lin"].values.astype(np.float32)
    pw  = g["RUL_piecewise125"].values.astype(np.float32)
    for end in range(len(g)):
        Xtr.append(summarize(f[:end + 1])); y_lin.append(lin[end]); y_pw.append(pw[end])
Xtr = np.array(Xtr, np.float32)
y_lin, y_pw = np.array(y_lin, np.float32), np.array(y_pw, np.float32)

Xte = np.array([
    summarize(test_full[test_full["unit"] == u]
              .sort_values("cycle")[FINAL_FEATURES_FD001].values.astype(np.float32))
    for u in units_test
], np.float32)
print(f"Built train {Xtr.shape} and test {Xte.shape} in {time.time()-t0:.0f}s")
print(f"Linear target max {y_lin.max():.0f} | piecewise target max {y_pw.max():.0f}")
print(f"Official test RUL: min {y_true.min():.0f}, max {y_true.max():.0f}\n")

def constrain(p, upper):
    """Lower-bound at zero; apply the upper bound only for the piecewise target."""
    return np.clip(p, 0, upper) if upper is not None else np.maximum(p, 0)

rows = []
for label, y_tr_, upper in [("linear", y_lin, None), ("piecewise", y_pw, CAP)]:
    # Linear regression is deterministic: one fit
    sc = MinMaxScaler().fit(Xtr)
    lr = LinearRegression().fit(sc.transform(Xtr), y_tr_)
    p = constrain(lr.predict(sc.transform(Xte)), upper)
    rows.append({"model": "LR", "target": label,
                 "mae": mean_absolute_error(y_true, p), "mae_sd": np.nan,
                 "rmse": np.sqrt(mean_squared_error(y_true, p)),
                 "nasa": nasa_score(y_true, p), "n_seeds": 1})

    # XGBoost: five seeds, reported as the mean across runs
    per_seed = []
    for seed in SEEDS:
        params = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8,
                      random_state=seed, n_jobs=-1, objective="reg:squarederror")
        m = xgb.XGBRegressor(**params).fit(Xtr, y_tr_)
        p = constrain(m.predict(Xte), upper)
        per_seed.append({"mae": mean_absolute_error(y_true, p),
                         "rmse": np.sqrt(mean_squared_error(y_true, p)),
                         "nasa": nasa_score(y_true, p)})
    d = pd.DataFrame(per_seed)
    rows.append({"model": "XGBoost", "target": label,
                 "mae": d["mae"].mean(), "mae_sd": d["mae"].std(),
                 "rmse": d["rmse"].mean(), "nasa": d["nasa"].mean(),
                 "n_seeds": len(SEEDS)})
    print(f"  {label:<10} XGBoost seeds: " +
          " ".join(f"{v:.2f}" for v in d["mae"]))

res = pd.DataFrame(rows)
print(f"\n{'Model':<10}{'Target':<12}{'MAE':>8}{'±SD':>7}{'RMSE':>8}{'NASA':>12}")
print("-" * 57)
for _, r in res.iterrows():
    sd = f"{r.mae_sd:.2f}" if not np.isnan(r.mae_sd) else "det."
    print(f"{r.model:<10}{r.target:<12}{r.mae:>8.2f}{sd:>7}{r.rmse:>8.2f}{r.nasa:>12.2f}")

print("\nChange from linear to piecewise:")
for mdl in ["LR", "XGBoost"]:
    s = res[res.model == mdl].set_index("target")
    for met in ["mae", "rmse", "nasa"]:
        a, b = s.loc["linear", met], s.loc["piecewise", met]
        print(f"  {mdl:<8} {met.upper():>5}: {a:9.2f} -> {b:8.2f}  "
              f"({100*(b-a)/a:+6.1f} %, absolute {a-b:+.2f})")

res.to_csv(os.path.join(RESULTS_DIR, "fd001_target_definition_final.csv"), index=False)
print("\nSaved: fd001_target_definition_final.csv")

In [ ]:
# ============================================================
# NB10 — Grouped permutation importance for the final XGBoost pipeline
# Change from the earlier version: importances are obtained by permuting
# a group of columns jointly rather than by summing single-column values.
# Permutation importances are not additive, and the five statistics of one
# sensor are correlated, so summing them would overstate or understate the
# group. Each group is permuted with one shared row ordering.
# The result is a conditional measure: it reflects what this fitted model
# relies on given the other inputs, not the information a feature carries
# in isolation.
# ============================================================
import os, sys, time
import numpy as np, pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
FIGURES_DIR   = os.path.join(PROJECT_ROOT, "figures")
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
from preprocessing import FINAL_FEATURES_FD001

CAP, SEED, N_REPEATS = 125, 42, 10
STATS = ["mean", "std", "slope", "last", "delta"]
XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=SEED, n_jobs=-1, objective="reg:squarederror")

def summarize(w):
    n = w.shape[0]; x = np.arange(n); feats = []
    for j in range(w.shape[1]):
        c = w[:, j]
        slope = np.polyfit(x, c, 1)[0] if n >= 2 else 0.0
        feats.extend([c.mean(), c.std(), slope, c[-1], c[-1] - c[0]])
    return np.array(feats, dtype=np.float32)

feat_names = [f"{s}_{st}" for s in FINAL_FEATURES_FD001 for st in STATS]

train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
units_test = test_last["unit"].values
y_true = test_last["RUL_true"].values

t0 = time.time()
Xtr, ytr = [], []
for unit, g in train.groupby("unit"):
    g = g.sort_values("cycle")
    f = g[FINAL_FEATURES_FD001].values.astype(np.float32)
    lab = g["RUL_piecewise125"].values.astype(np.float32)
    for end in range(len(g)):
        Xtr.append(summarize(f[:end + 1])); ytr.append(lab[end])
Xtr, ytr = np.array(Xtr, np.float32), np.array(ytr, np.float32)

Xte = np.array([
    summarize(test_full[test_full["unit"] == u]
              .sort_values("cycle")[FINAL_FEATURES_FD001].values.astype(np.float32))
    for u in units_test
], np.float32)
print(f"Built in {time.time()-t0:.0f}s | {Xte.shape[1]} inputs")

model = xgb.XGBRegressor(**XGB_PARAMS).fit(Xtr, ytr)
base = mean_absolute_error(y_true, np.clip(model.predict(Xte), 0, CAP))
print(f"Baseline test MAE (seed {SEED}): {base:.2f}\n")

def group_importance(cols, n_repeats=N_REPEATS):
    """Permute the given columns together, using one row ordering per
    repetition so the joint structure within the group is preserved."""
    rng = np.random.default_rng(SEED)
    deltas = []
    for _ in range(n_repeats):
        Xp = Xte.copy()
        order = rng.permutation(len(Xp))
        Xp[:, cols] = Xp[order][:, cols]
        mae = mean_absolute_error(y_true, np.clip(model.predict(Xp), 0, CAP))
        deltas.append(mae - base)
    return float(np.mean(deltas)), float(np.std(deltas, ddof=1))

# --- Groups: one per summary statistic (13 columns each) ---
stat_rows = []
for st in STATS:
    cols = [i for i, n in enumerate(feat_names) if n.endswith("_" + st)]
    m, s = group_importance(cols)
    stat_rows.append({"group": st, "n_columns": len(cols),
                      "importance": m, "std": s})
stat_df = pd.DataFrame(stat_rows).sort_values("importance", ascending=False)

print("By summary statistic (all thirteen columns permuted jointly):")
for _, r in stat_df.iterrows():
    print(f"  {r.group:<8} {r.importance:7.3f} ± {r['std']:5.3f}")

# --- Groups: one per sensor (5 columns each) ---
sens_rows = []
for s_ in FINAL_FEATURES_FD001:
    cols = [i for i, n in enumerate(feat_names) if n.startswith(s_ + "_")]
    m, s = group_importance(cols)
    sens_rows.append({"group": s_, "n_columns": len(cols),
                      "importance": m, "std": s})
sens_df = pd.DataFrame(sens_rows).sort_values("importance", ascending=False)

print("\nBy sensor (all five summaries permuted jointly):")
for _, r in sens_df.head(6).iterrows():
    print(f"  {r.group:<6} {r.importance:7.3f} ± {r['std']:5.3f}")

# --- Individual columns, for the leading features ---
ind_rows = []
for i, n in enumerate(feat_names):
    m, s = group_importance([i], n_repeats=5)
    ind_rows.append({"feature": n, "importance": m, "std": s})
ind_df = pd.DataFrame(ind_rows).sort_values("importance", ascending=False)

print("\nTop ten individual features:")
for _, r in ind_df.head(10).iterrows():
    print(f"  {r.feature:<14} {r.importance:6.3f} ± {r['std']:5.3f}")

for df_, name in [(stat_df, "statistic"), (sens_df, "sensor"), (ind_df, "individual")]:
    df_.to_csv(os.path.join(RESULTS_DIR, f"fd001_importance_{name}.csv"), index=False)

# --- Figure ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5),
                               gridspec_kw={"width_ratios": [1.6, 1]})
top = ind_df.head(15).iloc[::-1]
ax1.barh(top["feature"], top["importance"], xerr=top["std"],
         color="#2E75B6", edgecolor="black", linewidth=0.6, capsize=3)
ax1.set_xlabel("Increase in test MAE when permuted (cycles)")
ax1.grid(True, axis="x", alpha=0.3); ax1.set_axisbelow(True)

sd = stat_df.iloc[::-1]
ax2.barh(sd["group"], sd["importance"], xerr=sd["std"],
         color="#C55A11", edgecolor="black", linewidth=0.6, capsize=3)
ax2.set_xlabel("Increase in test MAE, group permuted jointly (cycles)")
ax2.grid(True, axis="x", alpha=0.3); ax2.set_axisbelow(True)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "fd001_xgb_importance_65.png"),
            dpi=300, bbox_inches="tight")
plt.show()
print("\nSaved: three CSVs + fd001_xgb_importance_65.png")

In [ ]:
# ============================================================
# NB10 — Target definition and output constraint, separated
# The earlier version changed two things at once: the training labels and
# the upper bound applied to the predictions. Capping a linear-target
# prediction at 125 does not make it a piecewise-trained model — it adds
# post-processing — so the two effects have to be separated to attribute
# anything to the labels alone.
# This cell stores the raw predictions once per target and seed, then
# evaluates each under both constraints. Nothing is refitted between the
# two constraint settings, so the difference is the constraint alone.
# Both are scored against the same unrestricted official test values.
# ============================================================
import os, sys, time
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
PRED_DIR      = os.path.join(RESULTS_DIR, "per_seed")
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
from preprocessing import FINAL_FEATURES_FD001
os.makedirs(PRED_DIR, exist_ok=True)

CAP, SEEDS = 125, [42, 43, 44, 45, 46]

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

def summarize(w):
    n = w.shape[0]; x = np.arange(n); feats = []
    for j in range(w.shape[1]):
        c = w[:, j]
        slope = np.polyfit(x, c, 1)[0] if n >= 2 else 0.0
        feats.extend([c.mean(), c.std(), slope, c[-1], c[-1] - c[0]])
    return np.array(feats, dtype=np.float32)

train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
train["RUL_lin"] = train.groupby("unit")["cycle"].transform("max") - train["cycle"]
units_test = test_last["unit"].values
y_true = test_last["RUL_true"].values

t0 = time.time()
Xtr, y_lin, y_pw = [], [], []
for unit, g in train.groupby("unit"):
    g = g.sort_values("cycle")
    f = g[FINAL_FEATURES_FD001].values.astype(np.float32)
    lin = g["RUL_lin"].values.astype(np.float32)
    pw  = g["RUL_piecewise125"].values.astype(np.float32)
    for end in range(len(g)):
        Xtr.append(summarize(f[:end + 1])); y_lin.append(lin[end]); y_pw.append(pw[end])
Xtr = np.array(Xtr, np.float32)
y_lin, y_pw = np.array(y_lin, np.float32), np.array(y_pw, np.float32)

Xte = np.array([
    summarize(test_full[test_full["unit"] == u]
              .sort_values("cycle")[FINAL_FEATURES_FD001].values.astype(np.float32))
    for u in units_test
], np.float32)
print(f"Built in {time.time()-t0:.0f}s | test RUL: min {y_true.min():.0f}, "
      f"max {y_true.max():.0f} | {(y_true > CAP).sum()} engines above the cap\n")

# --- Fit once per target and seed; keep the raw, unconstrained output ---
raw = {}                     # (model, target, seed) -> unconstrained prediction
for label, y_tr_ in [("linear", y_lin), ("piecewise", y_pw)]:
    sc = MinMaxScaler().fit(Xtr)
    lr = LinearRegression().fit(sc.transform(Xtr), y_tr_)
    raw[("LR", label, None)] = lr.predict(sc.transform(Xte))
    for seed in SEEDS:
        params = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8,
                      random_state=seed, n_jobs=-1, objective="reg:squarederror")
        m = xgb.XGBRegressor(**params).fit(Xtr, y_tr_)
        raw[("XGBoost", label, seed)] = m.predict(Xte)
    print(f"  fitted: {label}")

# --- Evaluate the stored predictions under both constraints ---
def score(p, upper):
    p = np.clip(p, 0, upper) if upper is not None else np.maximum(p, 0)
    return (mean_absolute_error(y_true, p),
            np.sqrt(mean_squared_error(y_true, p)),
            nasa_score(y_true, p))

rows = []
for label in ["linear", "piecewise"]:
    for cname, upper in [("none", None), ("cap 125", CAP)]:
        mae, rmse, ns = score(raw[("LR", label, None)], upper)
        rows.append({"model": "LR", "target": label, "constraint": cname,
                     "mae": mae, "mae_sd": np.nan, "rmse": rmse, "nasa": ns})

        vals = [score(raw[("XGBoost", label, s)], upper) for s in SEEDS]
        d = pd.DataFrame(vals, columns=["mae", "rmse", "nasa"])
        rows.append({"model": "XGBoost", "target": label, "constraint": cname,
                     "mae": d["mae"].mean(), "mae_sd": d["mae"].std(),
                     "rmse": d["rmse"].mean(), "nasa": d["nasa"].mean()})

res = pd.DataFrame(rows)

print(f"\n{'Model':<9}{'Target':<11}{'Constraint':<11}{'MAE':>8}{'±SD':>7}"
      f"{'RMSE':>8}{'NASA':>11}")
print("-" * 65)
for _, r in res.iterrows():
    sd = f"{r.mae_sd:.2f}" if not np.isnan(r.mae_sd) else "det."
    print(f"{r.model:<9}{r.target:<11}{r.constraint:<11}{r.mae:>8.2f}{sd:>7}"
          f"{r.rmse:>8.2f}{r.nasa:>11.2f}")

# --- Separate the two effects, holding the other fixed ---
print("\nEffect of the training labels, at a fixed constraint:")
for mdl in ["LR", "XGBoost"]:
    for cname in ["none", "cap 125"]:
        s = res[(res.model == mdl) & (res.constraint == cname)].set_index("target")
        a, b = s.loc["linear", "mae"], s.loc["piecewise", "mae"]
        print(f"  {mdl:<8} constraint {cname:<8}: {a:6.2f} -> {b:6.2f}  "
              f"({a-b:+6.2f} cycles)")

print("\nEffect of the output constraint, at fixed labels:")
for mdl in ["LR", "XGBoost"]:
    for label in ["linear", "piecewise"]:
        s = res[(res.model == mdl) & (res.target == label)].set_index("constraint")
        a, b = s.loc["none", "mae"], s.loc["cap 125", "mae"]
        print(f"  {mdl:<8} target {label:<10}: {a:6.2f} -> {b:6.2f}  "
              f"({a-b:+6.2f} cycles)")

res.to_csv(os.path.join(RESULTS_DIR, "fd001_target_and_constraint.csv"), index=False)

wide = pd.DataFrame({"unit": units_test, "RUL_true": y_true})
for (mdl, label, seed), p in raw.items():
    wide[f"{mdl}_{label}" + (f"_seed{seed}" if seed else "")] = p
wide.to_csv(os.path.join(PRED_DIR, "target_definition_raw.csv"), index=False)
print("\nSaved: fd001_target_and_constraint.csv + per_seed/target_definition_raw.csv")

In [ ]:
import pandas as pd, os
p = "{DATA_DIR}/../results/fd001_importance_sensor.csv"
d = pd.read_csv(p).sort_values("importance", ascending=False)
for _, r in d.iterrows():
    print(f"{r['group']:<6} {r['importance']:6.2f} {r['std']:6.2f}")